# vLLM GPU Sunucusu — Colab Runner

Bu defterin tek işi: [pqa](https://github.com/cxrbon16/pqa) pipeline'ının kullandığı QA üretici
modeli **vLLM** ile ayağa kaldırıp **Cloudflare Quick Tunnel** ile dışarıya (bu Colab dışındaki
bir makineye) açmak. Pipeline'ın kendisi (`fetch/passages/generate/filter/solve/band/publish`)
**burada değil**, tünelin ucundaki dış ortamda çalışıyor — bu defter sadece model + GPU sağlıyor.

Ollama'dan vLLM'e geçiş sebebi: Gemma 4'ün "thinking" (chain-of-thought) çıktısı Ollama'nın
OpenAI-uyumlu endpoint'inde `content` alanını boş bırakıyordu (token bütçesi düşünmeye gidiyordu).
vLLM'in `--reasoning-parser gemma4` bayrağı reasoning/content ayrımını doğru yapıyor.

**Önce:** `Çalışma zamanı > Çalışma zamanı türünü değiştir` menüsünden **G4** (NVIDIA RTX PRO 6000,
~96GB VRAM) seç — 31B modelin bf16'da çalışması için ~80GB gerekiyor. Sonra hücreleri sırayla
çalıştır, en sonda çıkan `https://xxxx.trycloudflare.com` URL'sini dış ortamdaki pipeline'ın
`config.yaml` → `base_url` alanına ver.

In [ ]:
!nvidia-smi

## 1. vLLM'i kur

Model: `google/gemma-4-31B-it` (gated **değil**, HF token gerekmiyor). Gemma 4 desteği vLLM'e
çıkış günü (2026-04-02) eklendi ama henüz stable release'e girmemiş olabilir — resmi rehber nightly
build kullanılmasını söylüyor ([kaynak](https://docs.vllm.ai/projects/recipes/en/stable/Google/Gemma4.html)).

Kurulum torch+CUDA wheel'leri içerdiği için büyük (birkaç GB) — disk kotanı önce kontrol et.

In [ ]:
!df -h /

In [ ]:
!pip install -q uv
!uv pip install --system -U vllm --pre \
  --extra-index-url https://wheels.vllm.ai/nightly/cu129 \
  --extra-index-url https://download.pytorch.org/whl/cu129 \
  --index-strategy unsafe-best-match

## 2. Sunucuyu başlat (model indirme + yükleme dahil, ilk çalıştırmada ~15-30 dk sürebilir)

`vllm serve`, HF'den `google/gemma-4-31B-it` ağırlıklarını (~63GB, safetensors) otomatik indirir,
sonra GPU'ya yükleyip OpenAI-uyumlu sunucuyu `:8000`'de açar. `--reasoning-parser gemma4` thinking
içeriğini ayrı bir alana (`reasoning_content`) koyar, asıl cevap `content`'te düzgün çıkar.

In [ ]:
import subprocess, time, urllib.request

log_path = "/content/vllm_serve.log"
log_file = open(log_path, "w")
vllm_proc = subprocess.Popen(
    [
        "vllm", "serve", "google/gemma-4-31B-it",
        "--trust-remote-code",
        "--reasoning-parser", "gemma4",
        "--max-model-len", "8192",
        "--gpu-memory-utilization", "0.90",
        "--host", "0.0.0.0",
        "--port", "8000",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    # Colab'da bir hücreyi durdurmak (SIGINT) aynı process group'taki arka plan sürecini de
    # öldürebilir — start_new_session bunu ayrı bir session'a alıp bu sorundan korur.
    start_new_session=True,
)
print("vllm serve pid:", vllm_proc.pid, "| log:", log_path)
print("İndirme + yükleme sürüyor, ilerlemeyi izlemek için bir sonraki hücreyi çalıştır.")

In [ ]:
# İlerlemeyi izle: sunucu hazır olana kadar bu hücreyi tekrar tekrar çalıştırabilirsin.
# "Uvicorn running on http://0.0.0.0:8000" satırı görününce sunucu hazırdır.
!tail -n 30 /content/vllm_serve.log

In [ ]:
import time, urllib.request

for _ in range(180):  # ~30 dk'ya kadar bekle
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=3)
        print("vLLM sunucusu hazır (http://localhost:8000).")
        break
    except Exception:
        if vllm_proc.poll() is not None:
            raise RuntimeError(
                f"vllm serve çöktü (çıkış kodu {vllm_proc.returncode}). Son log:\n"
                + "".join(open(log_path).readlines()[-40:])
            )
        time.sleep(10)
else:
    raise RuntimeError("30 dakikada hazır olmadı, log dosyasını kontrol et: " + log_path)

## 3. GPU kullanıldığını doğrula ve hızlı bir istekle test et

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
!curl -s http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "google/gemma-4-31B-it", "messages": [{"role": "user", "content": "Türkiye'nin başkenti neresidir? Tek kelimeyle cevap ver."}], "max_tokens": 500}' \
  | python3 -m json.tool

`choices[0].message.content` alanında **"Ankara"** gibi düzgün bir cevap görmelisin (boş değil).
Eğer varsa `choices[0].message.reasoning_content` alanı modelin iç muhakemesidir — bunu `content`'ten
ayrı tuttuğu için asıl sorunumuz çözülmüş demektir.

**Daha küçük GPU (T4/L4) kullanıyorsan:** `google/gemma-4-31B-it` (~80GB VRAM) T4/L4'te (16-24GB)
sığmaz. Yukarıdaki serve komutunda model adını `google/gemma-4-E4B-it` ile değiştir (~8B, çok daha
az VRAM gerektirir).

## 4. Tüneli aç

`localhost:8000`'i (vLLM) genel bir URL'ye proxy'ler — hesap/kayıt gerektirmez. Bu hücreyi
**sunucu hazır olduktan sonra** çalıştır.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
import subprocess, time, re

tunnel_log = "/content/cloudflared.log"
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT,
    start_new_session=True,  # hücre durdurulsa bile tünel ayakta kalsın
)

url = None
for _ in range(20):
    time.sleep(1)
    log = open(tunnel_log).read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if m:
        url = m.group(0)
        break

if url:
    print("Tünel URL'si (dış pipeline'ın config.yaml -> base_url alanına bunu ver):", url)
    print("örnek: " + url + "/v1")
else:
    print("URL 20 saniyede çıkmadı, log:\n" + open(tunnel_log).read())

## 5. Canlı tut

Bu sekmeyi kapatma / Colab'ı boşta bırakma — hem `vllm serve` hem `cloudflared` bu çalışma zamanı
sonlanınca ölür. Bağlantı kopar/URL değişirse bu hücreyi tekrar çalıştırıp yeni URL'yi dış
pipeline'a ilet:
```bash
!curl -s http://localhost:8000/health && echo OK   # sunucu hâlâ ayakta mı
!pgrep -af cloudflared                              # tünel hâlâ ayakta mı
```